# Análise de Tempo de Acesso de Leads
### Objetivo: Esse pipeline tem como objetivo definir um protocolo de tratamento dos dados de Compra dos Alunos do CDF de forma a definir quanto tempo de Acesso para cada produto eles possuem.

In [1]:
# Importações básicas
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

# Adiciona src ao path
sys.path.append('../src')

#PRODUCT Ids
CRONOGRAMA_ID = '3310674'

#Subdomains do Hotmart Club
CRONOGRAMA_SUBDOMAIN = 'cronogramadosfluentes-xwamel'

#Aulas de Conversação 
AULAS_DE_CONVERSACAO_SUBDOMAIN = 'aulasdeconversacao-vpqdjd'

#Ingles Express Travel
INGLES_EXPRESS_TRAVEL_SUBDOMAIN = 'inglesexpresstravel'

#Intensivão do Inglês
INTENSIVAO_DO_INGLES_BASICO_SUBDOMAIN = 'intensivaodoinglesbasico'

#Aulas de Conversação Particulares
AULAS_DE_CONVERSACAO_PARTICULARES_SUBDOMAIN = 'aulasdeconversacao-avjbpp'

# Utilitários de dados
from data_utils import (
    load_raw_data,
    save_processed_data,
    remove_duplicates,
    handle_missing_values,
    detect_outliers,
    normalize_column,
    process_phone_string,
    process_phone_number,
    clean_and_lower_column,
    flatten_list_to_df, 
    timestamp_ms_to_datetime
)

# Utilitários SQL
from sql_utils import DatabaseConnection as Dbc, load_query_from_file

# Utilitários de visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Utilitários de API
from api_utils import (
    make_request,
    get_json,
    post_json,
    paginated_request,
    response_to_dataframe
)

# utilitários hotmart
from hotmart_utils import Hotmart

# utilitários tmb
from tmb_utils import TMB   

# Configurações pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Load Database Driver
db = Dbc()

# Inicializar API Hotmart
hotmart = Hotmart()

# Inicializar API TMB
tmb = TMB()

print('✓ Importações concluídas com sucesso!')

✓ Importações concluídas com sucesso!


# Buscando Datasets de Compras da Hotmart

In [2]:
df_hotmart_pt_1 = flatten_list_to_df(hotmart.get_sales_history(
    start_date='2023-11-01',
    end_date='2024-12-31',
))

df_hotmart_pt_2 = flatten_list_to_df(hotmart.get_sales_history(
    start_date='2025-01-01',
    end_date='2025-12-31',
))

df_hotmart = pd.concat([df_hotmart_pt_1, df_hotmart_pt_2])

"""
Cell generated by Data Wrangler.
"""
def clean_data(df_hotmart):
    # Filter rows based on column: 'purchase_recurrency_number'
    df_hotmart = df_hotmart[(df_hotmart['purchase_recurrency_number'] == 1) | (df_hotmart['purchase_recurrency_number'].isna())]
    # Drop columns: 'purchase_hotmart_fee_currency_code', 'purchase_hotmart_fee_total' and 22 other columns
    df_hotmart = df_hotmart.drop(columns=['purchase_hotmart_fee_currency_code', 'purchase_hotmart_fee_total', 'purchase_hotmart_fee_fixed', 'purchase_hotmart_fee_percentage', 'purchase_hotmart_fee_base', 'purchase_is_subscription', 'purchase_offer_payment_mode', 'purchase_tracking_source', 'purchase_payment_type', 'purchase_payment_installments_number', 'purchase_commission_as', 'purchase_warranty_expire_date', 'purchase_order_date', 'purchase_price_value', 'purchase_price_currency_code', 'producer_ucode', 'product_id', 'producer_name', 'buyer_ucode', 'purchase_tracking_source_sck', 'purchase_recurrency_number', 'purchase_tracking_external_code', 'purchase_payment_method', 'purchase_status'])
    # Rename column 'purchase_transaction' to 'transaction'
    df_hotmart = df_hotmart.rename(columns={'purchase_transaction': 'transaction'})
    # Rename column 'purchase_offer_code' to 'offer_code'
    df_hotmart = df_hotmart.rename(columns={'purchase_offer_code': 'offer_code'})
    # Rename column 'purchase_approved_date' to 'approved_at'
    df_hotmart = df_hotmart.rename(columns={'purchase_approved_date': 'approved_at'})
    # Rename column 'buyer_name' to 'name'
    df_hotmart = df_hotmart.rename(columns={'buyer_name': 'name'})
    # Rename column 'buyer_email' to 'email'
    df_hotmart = df_hotmart.rename(columns={'buyer_email': 'email'})
    # Rename column 'approved_at' to 'approval_date'
    df_hotmart = df_hotmart.rename(columns={'approved_at': 'approval_date'})
    df_hotmart['approval_date'] = timestamp_ms_to_datetime(df_hotmart, 'approval_date')
    return df_hotmart

df_hotmart = clean_data(df_hotmart.copy())
df_hotmart.head()

Processando página 2 do histórico de vendas... (Total de itens: 50)

Timeout na requisição. Tentativa 1/3


Processando página 5 do histórico de vendas... (Total de itens: 200)

Timeout na requisição. Tentativa 1/3


✓ Concluído: 387 página(s) processada(s). Total de 19335 itens obtidos.)
✓ Concluído: 367 página(s) processada(s). Total de 18310 itens obtidos.)


,product_name,email,name,approval_date,transaction,offer_code
0,Cronograma dos Fluentes,oseildopedrosa@hotmail.com,Oseildo Pedrosa Josiele Noronha,2023-11-01 13:48:19,HP3359038232,zyvtnv0d
1,Manual Prático de Conversação em Inglês,oseildopedrosa@hotmail.com,Oseildo Pedrosa Josiele Noronha,2023-11-01 13:48:22,HP3050854363,g1d6dz6j
2,Cronograma dos Fluentes,georgiamattei@hotmail.com,Geórgia Mattei,2023-11-01 15:08:41,HP1200458985,zyvtnv0d
3,Cronograma dos Fluentes,osmarina.pessoa@gmail.com,Osmarina dos Anjos Pessoa,2023-11-01 16:01:33,HP3570912282,zyvtnv0d
4,Cronograma dos Fluentes,juliano_rdf@hotmail.com,Juliano Rodrigues de Freitas,2023-11-01 16:02:48,HP3632472106,zyvtnv0d


In [3]:
df_tmb = flatten_list_to_df(tmb.get_pedidos(page_size=1000))

"""
Cell generated by Data Wrangler.
"""
def clean_data(df_tmb):
    # Filter rows based on column: 'status_pedido'
    df_tmb = df_tmb[df_tmb['status_pedido'] == "Efetivado"]
    # Drop columns: 'produtor', 'status_pedido' and 29 other columns
    df_tmb = df_tmb.drop(columns=['produtor', 'status_pedido', 'documento', 'valor_total', 'parcelas', 'valor_entrada', 'valor_parcela', 'taxa_administracao', 'endereco_completo', 'criado_em', 'telefone', 'produto_id', 'nascimento', 'produtor_id', 'utm_source', 'utm_medium', 'utm_content', 'utm_campaign', 'utm_last_source', 'melhor_dia_pagamento', 'utm_last_medium', 'utm_last_content', 'utm_last_campaign', 'pais', 'cep', 'endereco_estado', 'endereco_cidade', 'endereco_bairro', 'endereco_logradouro', 'endereco_numero', 'endereco_complemento'])
    # Rename column 'lancamento' to 'offer_code'
    df_tmb = df_tmb.rename(columns={'lancamento': 'offer_code'})
    # Rename column 'pedido_id' to 'transaction'
    df_tmb = df_tmb.rename(columns={'pedido_id': 'transaction'})
    # Rename column 'cliente' to 'name'
    df_tmb = df_tmb.rename(columns={'cliente': 'name'})
    # Rename column 'data_efetivado' to 'approval_date'
    df_tmb = df_tmb.rename(columns={'data_efetivado': 'approval_date'})
    # Change column type to datetime64[ns] for column: 'approval_date'
    df_tmb = df_tmb.astype({'approval_date': 'datetime64[ns]'})
    return df_tmb

df_tmb = clean_data(df_tmb.copy())
df_tmb.head()

✓ Concluído: 1 página(s) processada(s). Total de 100 pedidos obtidos.


,offer_code,transaction,name,email,approval_date,status_financeiro
0,COM-BF25-alunos-Oferta Negociação,1918767,Matheus Ruas,matheusruas@hotmail.com,2025-12-02 18:53:01.355239,Adimplente
1,COM-BF25-alunos-Oferta Negociação,1917023,KLEBER FERREIRA DO VALE,klebervaleadvogado@outlook.com,2025-12-02 19:00:26.471830,Adimplente
2,COM-BF25-alunos-Oferta Negociação,1915079,Kyara Costa de Azevedo,kyky338.157@gmail.com,2025-12-01 10:55:35.356717,Adimplente
3,COM-BF25-Oferta Downsell-3,1914968,Bruno Luis Reis Intaschi,brunointaschi@hotmail.com,2025-12-01 11:21:29.853887,Adimplente
4,COM-BF25-Oferta Downsell-3,1914804,Joana Darc Santos Almeida Oliveira,oliveiragiseli071@gmail.com,2025-12-01 09:30:20.814242,Adimplente


# Dataframe de Compras Totais

In [4]:
df = pd.concat([df_hotmart, df_tmb])
df.head()


,product_name,email,name,approval_date,transaction,offer_code,status_financeiro
0,Cronograma dos Fluentes,oseildopedrosa@hotmail.com,Oseildo Pedrosa Josiele Noronha,2023-11-01 13:48:19,HP3359038232,zyvtnv0d,NaN
1,Manual Prático de Conversação em Inglês,oseildopedrosa@hotmail.com,Oseildo Pedrosa Josiele Noronha,2023-11-01 13:48:22,HP3050854363,g1d6dz6j,NaN
2,Cronograma dos Fluentes,georgiamattei@hotmail.com,Geórgia Mattei,2023-11-01 15:08:41,HP1200458985,zyvtnv0d,NaN
3,Cronograma dos Fluentes,osmarina.pessoa@gmail.com,Osmarina dos Anjos Pessoa,2023-11-01 16:01:33,HP3570912282,zyvtnv0d,NaN
4,Cronograma dos Fluentes,juliano_rdf@hotmail.com,Juliano Rodrigues de Freitas,2023-11-01 16:02:48,HP3632472106,zyvtnv0d,NaN


In [5]:
# Agrupando por email e criando lista de objetos com datas da primeira e última compra, lista de compras e quantidade de compras
grouped_purchases = []

for email, grp in df.sort_values('approval_date').groupby('email'):
    purchases = grp.drop(columns=['email']).to_dict(orient='records')
    first_purchase_date = grp['approval_date'].min()
    last_purchase_date = grp['approval_date'].max()
    num_purchases = len(purchases)
    grouped_purchases.append({
        'email': email,
        'first_purchase': first_purchase_date,
        'last_purchase': last_purchase_date,
        'purchases': purchases,
        'num_purchases': num_purchases  # Adiciona o número de compras
    })

grouped_purchases

[{'email': ' anna.silva021@outlook.com',
  'first_purchase': Timestamp('2025-12-01 09:43:34.512324'),
  'last_purchase': Timestamp('2025-12-01 09:43:34.512324'),
  'purchases': [{'product_name': nan,
    'name': 'Ana Cláudia Barata da Silva ',
    'approval_date': Timestamp('2025-12-01 09:43:34.512324'),
    'transaction': 1914162,
    'offer_code': 'COM-BF25-alunos-Oferta Live',
    'status_financeiro': 'Adimplente'}],
  'num_purchases': 1},
 {'email': '006fabi@gmail.com',
  'first_purchase': Timestamp('2024-01-17 01:44:15'),
  'last_purchase': Timestamp('2024-01-17 01:44:15'),
  'purchases': [{'product_name': 'Cronograma dos Fluentes ',
    'name': 'Fabíola Rodrigues Pereira',
    'approval_date': Timestamp('2024-01-17 01:44:15'),
    'transaction': 'HP1764838609',
    'offer_code': 'zyvtnv0d',
    'status_financeiro': nan}],
  'num_purchases': 1},
 {'email': '0311cintia@gmail.com',
  'first_purchase': Timestamp('2024-02-08 19:24:20'),
  'last_purchase': Timestamp('2024-02-08 19:24:2

# Carregando Ofertas e Entregáveis

In [6]:
df_offers_deliverables = load_raw_data('offers_deliverables.csv')
df_offers_deliverables = df_offers_deliverables.rename(columns={'offer_id': 'offer_code'})

# Criando Dataframe Accesses

In [7]:
from dateutil.relativedelta import relativedelta
from datetime import datetime

access_rows = []

now = pd.Timestamp.now()

# Etapa 1: Construir df_accesses normalmente (agora incluindo weekly_classes)
for buyer in grouped_purchases:
    email = buyer['email']
    for purchase in buyer['purchases']:
        offer_code = purchase['offer_code']
        offer = df_offers_deliverables[df_offers_deliverables['offer_code'] == offer_code]
        
        if not offer.empty:
            for _, deliverable in offer.iterrows():
                transaction = purchase.get('transaction', None)
                deliverable_name = deliverable.get('deliverable_name', None)
                duration_access = deliverable.get('duration_access', None)
                weekly_classes = deliverable.get('weekly_classes', 0)
                purchase_date = pd.to_datetime(purchase['approval_date'])

                # expiration_date
                if pd.isnull(duration_access):
                    expiration_date = None
                else:
                    try:
                        duration_access_int = int(duration_access)
                        try:
                            expiration_date = purchase_date + pd.DateOffset(months=duration_access_int)
                        except (OverflowError, pd.errors.OutOfBoundsDatetime):
                            expiration_date = pd.Timestamp.max
                    except Exception:
                        expiration_date = None

                # status
                if expiration_date is None:
                    status = 'Indeterminate'
                elif expiration_date < now:
                    status = 'Expired'
                else:
                    status = 'Active'
                
                access_rows.append({
                    'transaction': transaction,
                    'offer_code': offer_code,
                    'email': email,
                    'deliverable_name': deliverable_name,
                    'duration_access': duration_access,
                    'purchase_date': purchase_date,
                    'expiration_date': expiration_date,
                    'status': status,
                    'weekly_classes': weekly_classes
                })

df_accesses = pd.DataFrame(access_rows)

# Mantém Active / Indeterminate
df_valid_accesses = df_accesses[df_accesses['status'].isin(['Active', 'Indeterminate'])].copy()

# Auditoria - todas as transações
df_audit_all = df_accesses.copy()

consolidated_rows = []

def compute_consolidated_periods_custom(group):
    records = group.sort_values('purchase_date').to_dict('records')
    periods = []
    last_expiration = None

    for rec in records:
        dur = rec['duration_access']
        trans = rec['transaction']
        wclasses = rec['weekly_classes']

        if pd.isnull(dur) or dur == 0:
            proposed_start = rec['purchase_date']
            proposed_end = None
        else:
            duration_int = int(dur)

            if last_expiration is not None and pd.notnull(last_expiration) and rec['purchase_date'] < last_expiration:
                proposed_start = last_expiration
            else:
                proposed_start = rec['purchase_date']

            if pd.notnull(proposed_start):
                try:
                    proposed_end = proposed_start + pd.DateOffset(months=duration_int)
                except (OverflowError, pd.errors.OutOfBoundsDatetime):
                    proposed_end = pd.Timestamp.max
            else:
                proposed_end = None

        periods.append({
            'transaction': trans,
            'start': proposed_start,
            'end': proposed_end,
            'duration_access': dur,
            'purchase_date': rec['purchase_date'],
            'weekly_classes': wclasses
        })

        if proposed_end is not None and (last_expiration is None or proposed_end > last_expiration):
            last_expiration = proposed_end

    return periods


for (email, deliverable), group in df_valid_accesses.groupby(['email', 'deliverable_name']):
    group_sorted = group.sort_values('purchase_date')

    # períodos consolidados
    custom_periods = compute_consolidated_periods_custom(group_sorted)

    # mapa effective_start/end para auditoria
    effective_map = {
        p['transaction']: {
            'effective_start_date': p['start'],
            'effective_end_date': p['end']
        }
        for p in custom_periods if p['transaction'] is not None
    }

    audit_group = df_audit_all[
        (df_audit_all['email'] == email) &
        (df_audit_all['deliverable_name'] == deliverable)
    ].sort_values('purchase_date')

    transactions_auditoria = []
    for _, row in audit_group.iterrows():
        eff = effective_map.get(row['transaction'], {})
        transactions_auditoria.append({
            'transaction': row['transaction'],
            'offer_code': row['offer_code'],
            'original_purchase_date': row['purchase_date'],
            'original_expiration_date': row['expiration_date'],
            'effective_start_date': eff.get('effective_start_date'),
            'effective_end_date': eff.get('effective_end_date'),
            'duration_access': row['duration_access'],
            'status': row['status'],
            'weekly_classes': row.get('weekly_classes', 0)
        })

    # resumo histórico
    first_date = audit_group['purchase_date'].min()
    last_date = audit_group['purchase_date'].max()
    purchase_count = len(audit_group)

    if purchase_count > 1:
        renewal_purchase_dates = list(audit_group['purchase_date'].sort_values().iloc[1:])
    else:
        renewal_purchase_dates = []

    # maior data de expiração final
    effective_end_date = None
    for p in custom_periods:
        if pd.notnull(p['end']):
            if effective_end_date is None or p['end'] > effective_end_date:
                effective_end_date = p['end']

    # somatório weekly_classes
    weekly_classes_total = audit_group['weekly_classes'].fillna(0).sum()

    # ------------------------------------------------------------
    # NOVO: Tempo total de acesso em dias
    # ------------------------------------------------------------
    total_access_days = 0
    for p in custom_periods:
        if pd.notnull(p['start']) and pd.notnull(p['end']):
            delta_days = (p['end'] - p['start']).days
            if delta_days > 0:
                total_access_days += delta_days

    # função auxiliar para converter dias → meses aproximados
    def days_to_months(days):
        return round(days / 30, 2) if days is not None else None

    # tempo total em meses
    total_access_months = days_to_months(total_access_days)

    # tempo restante em meses
    if effective_end_date is not None:
        delta_remaining = (effective_end_date - now).days
        remaining_access_months = days_to_months(delta_remaining if delta_remaining > 0 else 0)
    else:
        remaining_access_months = None

    # ------------------------------------------------------------

    consolidated_rows.append({
        'email': email,
        'deliverable_name': deliverable,

        'first_date': first_date,
        'last_date': last_date,
        'purchase_count': purchase_count,
        'renewal_purchase_dates': renewal_purchase_dates,

        'effective_end_date': effective_end_date,

        'weekly_classes_total': weekly_classes_total,

        # novos campos
        'total_access_days': total_access_days,
        'total_access_months': total_access_months,
        'remaining_access_months': remaining_access_months,

        'period_purchase_transaction_audit': transactions_auditoria,
    })

df_accesses_consolidated = pd.DataFrame(consolidated_rows)
